In [1]:
import numpy as np
from pyscf import gto, sftda
from pyscf.sftda.uhf_sf import get_ab_sf
from pyscf.sftda.tools_td import transition_analyze, spin_square
import sys
sys.path.append('/home/peter/Research/ZeroExcitationEnergy/code/src')
sys.path.append('/public1/home/scg0213/Jobs_WT/ZeroExcitationEnergy/src')
from tools import extract_state

In [12]:
r_list = np.concatenate((np.linspace(0.4, 3.0, 53), np.linspace(3.2, 6.0, 15)))
fun_list = ['PBE']
bas = '6-311g*'

In [13]:
mol = gto.M(atom=f'H; F 1 {20.0}', basis=bas, charge=0, spin=2)
mf = mol.UKS(xc='PBE')
mf.kernel()

converged SCF energy = -100.151515723128  <S^2> = 2.0007768  2S+1 = 3.0005178


-100.15151572312834

In [14]:
a, b = get_ab_sf(mf, collinear_samples=50)
A_baba, A_abab = a
B_baab, B_abba = b

n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                        [-B_baab_2d,-A_baba_2d]])

eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

idxt = eigenvals.real.argsort()
eigenvals = eigenvals[idxt]
eigenvecs = eigenvecs[:, idxt]

norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

sfd_eigenvals = eigenvals[norms>0]
sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

def norm_xy(z):
    x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
    y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
    norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
    norm = np.sqrt(1./norm)
    return ((0,x*norm), (y*norm,0))

mftd1 = sftda.TDA_SF(mf)
mftd1.nstates = len(sfd_eigenvals)
mftd1.extype = 1
mftd1.collinear_samples = 50
mftd1.e = sfd_eigenvals
mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

target = extract_state(mf, mftd1, tdtype='TDDFT', Smin=0.0, Smax=0.9)[0]

print(transition_analyze(mf, mftd1, mftd1.e[target], mftd1.xy[target], tdtype='TDDFT'))

50
0-th state with excitation energy = -1.1916, S^2 = 0.0030 Norms: 1.000@(6, 1), 0.000@(5, 1)
Excited energy : -1.1915877123763658 eV.
<S2> value : 0.0030066069010206142.
The main and second norm:
1.0 7.408581799714605e-29
The main and second norm to orbital pair:
(6.0, 1.0) (5.0, 1.0)


None


In [15]:
mol.ao_labels()

['0 H 1s    ',
 '0 H 2s    ',
 '0 H 3s    ',
 '1 F 1s    ',
 '1 F 2s    ',
 '1 F 3s    ',
 '1 F 4s    ',
 '1 F 2px   ',
 '1 F 2py   ',
 '1 F 2pz   ',
 '1 F 3px   ',
 '1 F 3py   ',
 '1 F 3pz   ',
 '1 F 4px   ',
 '1 F 4py   ',
 '1 F 4pz   ',
 '1 F 3dxy  ',
 '1 F 3dyz  ',
 '1 F 3dz^2 ',
 '1 F 3dxz  ',
 '1 F 3dx2-y2']

In [16]:
mf.mo_coeff[0][:, 5]

array([ 2.51104208e-01,  4.59685065e-01,  4.21539429e-01,  2.40215218e-56,
       -9.27596284e-56,  3.69105551e-55, -4.56175962e-55,  6.95383525e-55,
        1.23964294e-55, -1.23741683e-59, -1.89689336e-54, -3.38154469e-55,
        3.37547223e-59,  2.39577134e-54,  4.27088206e-55, -4.26321256e-59,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00])

In [17]:
mf.mo_coeff[1][:, 4]

array([-8.09956809e-56,  4.64067485e-55, -6.60515982e-55, -1.48139927e-16,
       -3.08638658e-17,  3.05412790e-16,  4.39154815e-16,  3.82270283e-08,
        3.20451902e-01,  1.67829550e-02,  5.34226410e-08,  4.47795330e-01,
        2.34541569e-02,  5.47708071e-08,  4.59095939e-01,  2.40460622e-02,
        2.18800920e-11,  1.34265424e-17, -5.26684408e-17,  1.14647948e-12,
       -2.98398918e-16])

In [18]:
mf.analyze()

**** MO energy ****
                             alpha | beta                alpha | beta
MO #1   energy= -24.3226284105829  | -24.2940660961397  occ= 1 | 1
MO #2   energy= -1.10256136276705  | -1.0292477975714   occ= 1 | 1
MO #3   energy= -0.473016661528193 | -0.354091823069925 occ= 1 | 1
MO #4   energy= -0.382683687265335 | -0.354090888837843 occ= 1 | 1
MO #5   energy= -0.382683636052065 | -0.321404630586571 occ= 1 | 0
MO #6   energy= -0.277614586004235 | 0.883974680716058  occ= 1 | 0
MO #7   energy= 0.215451525012396  | 0.883975348833043  occ= 0 | 0
MO #8   energy= 0.81795251429834   | 0.904627518244974  occ= 0 | 0
MO #9   energy= 0.867983796449906  | 1.00219950596973   occ= 0 | 0
MO #10  energy= 0.867983811520336  | 1.2723647344375    occ= 0 | 0
MO #11  energy= 0.957232242126163  | 3.65950126271267   occ= 0 | 0
MO #12  energy= 2.07327898047296   | 3.65950234898382   occ= 0 | 0
MO #13  energy= 3.54715641929801   | 3.66602930160227   occ= 0 | 0
MO #14  energy= 3.57070481902074   | 3.

(((array([9.92522990e-01, 6.03125051e-03, 1.44575990e-03, 9.99984152e-01,
          9.99719027e-01, 5.71344272e-05, 2.74826041e-06, 9.99937957e-01,
          9.98875223e-01, 9.99935052e-01, 5.59623842e-05, 1.03374471e-03,
          5.86347270e-05, 6.08074399e-06, 9.10321719e-05, 6.31288542e-06,
          1.00256341e-17, 1.93274615e-06, 5.82696715e-05, 2.82860986e-20,
          1.76736142e-04]),
   array([7.93912723e-109, 4.06466214e-108, 1.12176332e-110, 9.99988069e-001,
          9.99581755e-001, 4.08199507e-004, 2.21516195e-006, 9.99994939e-001,
          2.73573256e-003, 9.97259209e-001, 4.39033429e-006, 1.19042997e-008,
          4.37606579e-006, 6.70827347e-007, 1.76093355e-009, 6.68354303e-007,
          8.36588570e-019, 1.61297834e-007, 4.85977065e-006, 3.08255986e-021,
          1.47395209e-005])),
  array([-2.22044605e-16,  0.00000000e+00])),
 array([-4.33646943e-09,  2.84749525e-15,  7.18109756e-16]))

In [3]:
for fun in fun_list:
    print(f'Calculating {fun}...')
    refs2_list = []
    s2_list = []
    td_list = []
    for r in r_list:
        mol = gto.M(atom=f'H; F 1 {r}', basis=bas, charge=0, spin=2)
        mf = mol.UKS(xc=fun)
        mf.kernel()

        refs2_list.append(mf.spin_square())

        a, b = get_ab_sf(mf, collinear_samples=50)
        A_baba, A_abab = a
        B_baab, B_abba = b

        n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
        n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

        A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
        B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
        B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
        A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

        Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                                [-B_baab_2d,-A_baba_2d]])

        eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

        idxt = eigenvals.real.argsort()
        eigenvals = eigenvals[idxt]
        eigenvecs = eigenvecs[:, idxt]

        norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
        norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

        sfd_eigenvals = eigenvals[norms>0]
        sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

        def norm_xy(z):
            x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
            y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
            norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
            norm = np.sqrt(1./norm)
            return ((0,x*norm), (y*norm,0))

        mftd1 = sftda.TDA_SF(mf)
        mftd1.nstates = len(sfd_eigenvals)
        mftd1.extype = 1
        mftd1.collinear_samples = 50
        mftd1.e = sfd_eigenvals
        mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

        target = extract_state(mf, mftd1, tdtype='TDDFT', Smin=0.0, Smax=0.9)[0]

        print(transition_analyze(mf, mftd1, mftd1.e[target], mftd1.xy[target], tdtype='TDDFT'))
        td_list.append(mftd1.e[target] + mf.e_tot)
        s2_list.append(spin_square(mf, mftd1.xy[target], extype=1))
    np.save('r.npy', r_list)
    np.save(f'{fun}_e.npy', td_list)
    np.save(f'{fun}_Refs2.npy', refs2_list)
    np.save(f'{fun}_s2.npy', s2_list)
    print('-----------------------')

Calculating PBE0...
converged SCF energy = -98.2182007423806  <S^2> = 2.000906  2S+1 = 3.0006039
50
1-th state with excitation energy = -0.5550, S^2 = 0.007 Norms: 1.00@(6, 1), 0.00@(2, 1)
2-th state with excitation energy = 0.0119, S^2 = 0.002 Norms: 0.50@(6, 2), 0.49@(4, 1)
3-th state with excitation energy = 0.8758, S^2 = 0.009 Norms: 0.99@(4, 2), 0.00@(2, 1)
Excited energy : -15.101711211316626 eV.
<S2> value : 0.007038893796242451.
The main and second norm:
0.9996594627460738 0.00029157966145822704
The main and second norm to orbital pair:
(6.0, 1.0) (2.0, 1.0)


None
converged SCF energy = -98.783984073903  <S^2> = 2.0009238  2S+1 = 3.0006158
50
1-th state with excitation energy = -0.5422, S^2 = 0.007 Norms: 1.00@(6, 1), 0.00@(2, 1)
2-th state with excitation energy = 0.0119, S^2 = 0.002 Norms: 0.51@(6, 2), 0.49@(4, 1)
3-th state with excitation energy = 0.8628, S^2 = 0.011 Norms: 0.99@(4, 2), 0.01@(2, 1)
Excited energy : -14.753561709266377 eV.
<S2> value : 0.00708584936933665.


In [ ]:
import numpy as np
from pyscf import gto, fci

hf_list = []
fci_list = []
for r in r_list:
    mol = gto.M(atom=f'H; F 1 {r}', basis=bas, charge=0, spin=0)
    mf = mol.RHF()
    mf.max_memory = 80000
    mf.kernel()
    hf_list.append(mf.e_tot)
    myci = fci.FCI(mf)
    myci.max_memory = 80000
    myci.kernel()
    fci_list.append(myci.e_tot)
    print(f'r = {r:.2f}, HF energy = {mf.e_tot:.6f}, FCI energy = {myci.e_tot:.6f}')
    np.save('HF_e.npy', hf_list)
    np.save('FCI_e.npy', fci_list)

converged SCF energy = -98.3620601112554
r = 0.40, HF energy = -98.362060, FCI energy = -98.521693
converged SCF energy = -98.9275115116962
r = 0.45, HF energy = -98.927512, FCI energy = -99.090347
converged SCF energy = -99.3060513726074


In [2]:
import numpy as np

np.load('B3LYP_e.npy') - np.load('B3LYP_e.npy')[-1]

array([ 1.31757192e+00,  7.65997579e-01,  3.96111950e-01,  1.47118383e-01,
       -2.01497006e-02, -1.31352857e-01, -2.03658186e-01, -2.48759623e-01,
       -2.74764313e-01, -2.87476364e-01, -2.91087342e-01, -2.88649028e-01,
       -2.82332703e-01, -2.73642743e-01, -2.63532273e-01, -2.52646017e-01,
       -2.41354053e-01, -2.29912651e-01, -2.18512202e-01, -2.07247768e-01,
       -1.96217498e-01, -1.85519710e-01, -1.75186093e-01, -1.65280269e-01,
       -1.55825604e-01, -1.46863608e-01, -1.38393492e-01, -1.30419413e-01,
       -1.22945020e-01, -1.15979398e-01, -1.09485065e-01, -1.03448196e-01,
       -9.78462161e-02, -9.26639824e-02, -8.78697860e-02, -8.34511601e-02,
       -7.93881963e-02, -7.56467943e-02, -7.22041595e-02, -6.90513986e-02,
       -6.61646033e-02, -6.35467572e-02, -6.11447025e-02, -5.89752293e-02,
       -5.70099525e-02, -5.52410123e-02, -5.36638564e-02, -5.22489751e-02,
       -5.09799590e-02, -4.98819801e-02, -4.89063262e-02, -4.80580039e-02,
       -4.73242424e-02, -

In [3]:
np.load('SVWN_e.npy') - np.load('SVWN_e.npy')[-1]

array([ 1.39444787e+00,  8.41078934e-01,  4.69391359e-01,  2.18637383e-01,
        4.97686323e-02, -6.28269149e-02, -1.36260269e-01, -1.82205249e-01,
       -2.08827912e-01, -2.21994456e-01, -2.25990778e-01, -2.23940525e-01,
       -2.18064145e-01, -2.09877720e-01, -2.00374502e-01, -1.90181422e-01,
       -1.79684646e-01, -1.69120987e-01, -1.58639538e-01, -1.48338495e-01,
       -1.38288343e-01, -1.28543535e-01, -1.19146649e-01, -1.10133137e-01,
       -1.01529997e-01, -9.33571586e-02, -8.56285110e-02, -7.83523059e-02,
       -7.15317222e-02, -6.51652254e-02, -5.92450150e-02, -5.37608875e-02,
       -4.86990325e-02, -4.40447113e-02, -3.97808194e-02, -3.58892069e-02,
       -3.23499537e-02, -2.91419451e-02, -2.62434236e-02, -2.36325732e-02,
       -2.12872359e-02, -1.91856887e-02, -1.73063369e-02, -1.56290152e-02,
       -1.41338489e-02, -1.28024472e-02, -1.16169069e-02, -1.05615919e-02,
       -9.62157577e-03, -8.78214645e-03, -8.03389287e-03, -7.36282637e-03,
       -6.75981893e-03, -

In [1]:
import numpy as np
np.load('FCI_e.npy')[-1]

-100.08435307677912

In [3]:
bas = '6-311g*'
from pyscf import gto, fci
mol = gto.M(atom='F', charge=-1, spin=0, basis=bas)
mf = mol.RHF()
mf.max_memory = 80000
mf.kernel()
myci = fci.FCI(mf)
myci.max_memory = 80000
myci.kernel()

converged SCF energy = -99.4028924470123


(-99.63583449047809,
 FCIvector([[ 9.79844670e-01,  6.35354860e-04, -3.29046484e-04, ...,
             -7.79972519e-14,  5.61018011e-14, -1.95890809e-14],
            [ 6.35354860e-04, -3.63376031e-02,  1.60140478e-02, ...,
             -1.50833866e-08,  6.03610038e-10,  3.60254263e-08],
            [-3.29046484e-04,  1.60140478e-02, -1.37100774e-02, ...,
             -2.25427899e-08, -5.55679101e-11, -4.44729652e-09],
            ...,
            [-7.79972521e-14, -1.50833866e-08, -2.25427899e-08, ...,
             -9.24912618e-11,  7.59479990e-16, -2.19369389e-16],
            [ 5.61018012e-14,  6.03610038e-10, -5.55679101e-11, ...,
              7.59479990e-16, -9.25080399e-11,  3.13189893e-16],
            [-1.95890808e-14,  3.60254263e-08, -4.44729652e-09, ...,
             -2.19369389e-16,  3.13189893e-16, -7.15293952e-11]]))

In [4]:
bas = '6-311g*'
from pyscf import gto, fci
mol = gto.M(atom='F', charge=0, spin=1, basis=bas)
mf = mol.UHF()
mf.max_memory = 80000
mf.kernel()
myci = fci.FCI(mf)
myci.max_memory = 80000
myci.kernel()

converged SCF energy = -99.3968737176783  <S^2> = 0.75224337  2S+1 = 2.0022421


(-99.58454324672934,
 FCIvector([[ 9.84679138e-01,  3.43936256e-15,  8.33793933e-16, ...,
             -3.67816062e-07, -3.22223335e-20,  2.71439285e-22],
            [ 6.83408561e-17,  5.97203242e-03, -5.30810202e-03, ...,
             -4.91828311e-22,  8.35851356e-09,  3.42659390e-23],
            [-1.33922424e-16, -5.30810202e-03, -5.97203242e-03, ...,
             -6.77342709e-22, -1.82685652e-09,  4.62511669e-23],
            ...,
            [ 3.49668768e-22,  4.76922486e-08, -1.22602529e-07, ...,
              5.85824764e-23, -4.33960557e-11, -1.02621622e-24],
            [-4.74258255e-22, -2.20505743e-21, -5.89391893e-22, ...,
              2.01040718e-11,  1.10139793e-24,  7.24001324e-19],
            [-1.01503817e-22, -9.81492899e-22, -2.46291143e-22, ...,
              1.18500109e-11,  2.16489175e-24, -1.67452756e-11]]))

In [5]:
np.concatenate((np.load('FCI/Results/e.npy')[2:-2], np.load('FCI_e.npy')[-3:]))

array([ -99.99255583, -100.17840437, -100.25275563, -100.26742997,
       -100.27303293, -100.27243016, -100.26766426, -100.26018494,
       -100.25101045, -100.23017599, -100.2085328 , -100.1876888 ,
       -100.1684829 , -100.15139289, -100.13666905, -100.12437601,
       -100.11441992, -100.10658458, -100.10057686, -100.09607347,
       -100.09275993, -100.0903566 , -100.08863122, -100.08740076,
       -100.08652661, -100.08590673, -100.08546737, -100.08515578,
       -100.08477669, -100.08444009, -100.08440608, -100.08435487,
       -100.08435308])

In [21]:
mol = gto.M(atom=f'H; F 1 {20.0}', basis=bas, charge=0, spin=0)
mf = mol.RHF()
mf.max_memory = 80000
mf.kernel()
from pyscf import gto, fci
myci = fci.FCI(mf)
myci.max_memory = 110000
myci.kernel()

OSError: [Errno 28] Unable to synchronously create file (file write failed: time = Mon Jun 23 12:18:40 2025
, filename = '/tmp/tmpjq9vp6j_', file descriptor = 68, errno = 28, error message = 'No space left on device', buf = 0x139cce08, total write size = 96, bytes this sub-write = 96, bytes actually written = 18446744073709551615, offset = 0)